# CILC method

After having implemented the ILC method properly and checking that it is not enough to extract the $\mu$-distorted $y$ signal, I am taking a step further by $\textit{constraining}$ the ILC method, so now I will proceed with the CILC method.

This code must

1. Calculate the covariance matrix: $$ \widehat{\mathbf{R}}_{ij}(\bar{\ell}) = \frac{1}{N_{\bar{\ell}}} \sum_{\bar{\ell} - \Delta \bar{\ell}}^{\bar{\ell} + \Delta \bar{\ell}} \sum_{m} x_{\ell m, i} x_{\ell m, j}^{*} , $$
where $N_{\ell} = (\ell + \Delta \ell + 1)^2 - (\ell - \Delta \ell)^2$.


2. Compute the transposed weights: $$\underline{\omega^t} = \frac{(\underline{b^t} \cdot \underline{\underline{\widehat{R}^{-1}}} \cdot \underline{b}) \cdot \underline{a^t} \cdot \underline{\underline{\widehat{R}^{-1}}} - (\underline{a^t} \cdot \underline{\underline{\widehat{R}^{-1}}} \cdot \underline{b}) \cdot \underline{b^t} \cdot \underline{\underline{\widehat{R}^{-1}}} }{ (\underline{a^t} \cdot {\underline{\underline{\widehat{R}^{-1}}}} \cdot \underline{a}) (\underline{b^t} \cdot \underline{\underline{\widehat{R}^{-1}}} \cdot \underline{b}) - (\underline{a^t} \cdot {\underline{\underline{\widehat{R}^{-1}}}} \cdot \underline{b})^2 } $$

3. Apply the weights ($\underline{\omega^t}$) to the data ($d_i$).

---
Shouldn't we transpose $\underline{\omega^t}$ in order to apply it to the weights?
---
---

4. Get the output: $$\hat{y} = \Sigma_i \left( \omega_i \cdot d_i (p) \right), $$
and multiply it by $\mu$.

5. Marginalise over the $\widehat{\mu y}$ tSZ signal so that:
$$ <\widehat{\mu y}>^2 \text{is minimum,}$$
$$ \tilde{\underline{w^t}} \cdot \underline{h} = 1,$$
$$ \tilde{\underline{w^t}} \cdot \underline{g} = 0.$$

### Previous

In [1]:
### IMPORT PACKAGES ###

import healpy as hp
import numpy as np
import pandas as pd
import scipy as sp
import numba as nb
import matplotlib.pyplot as plt

import gc
from joblib import Parallel, delayed
from numba import njit, prange
from scipy.interpolate import *
from scipy import stats

from astropy.wcs import WCS
from reproject import reproject_from_healpix

In [2]:
### MEMORY MANAGEMENT ###

import gc # Garbage Collector

# Before starting a new run, clear previous big variables if they exist
if 'data_cube' in locals():
    del data_cube
if 'alms_list' in locals():
    del alms_list
if 'y_hat_harmonic' in locals():
    del y_hat_harmonic
if 'y_hat_pixel' in locals():
    del y_hat_pixel
if 'residuals_harmonic' in locals():
    del residuals_harmonic
if 'residuals_pixel' in locals():
    del residuals_pixel

gc.collect() # Manually trigger memory cleanup

32

In [3]:
### DEFINITIONS: CONSTANTS, FREQUENCIES, ETC. ###

# Nside
nside_work = 2048

# Conversion factor for Compton-y to muK
T_cmb_muK = 2.7255e6

# Planck frequencies in GHz --- kickoff diapo. 8
frequencies = [30, 44, 70, 100, 143, 217, 353, 545, 857]

# Reference noise
base_sensitivity_uk_arcmin = 30.0

# Theoretical (LCDM) mu-distortion value
#mu = 2e-8
mu = 2e-3

# Calculate area of pixel in arcmin^2 for the working resolution. As hp.nside2pixarea gives us the area in steradians, we convert it to arcmin^2 by multiplying by (180*60/pi)^2
pixel_area_arcmin2 = hp.nside2pixarea(nside_work) * (180*60/np.pi)**2

### Pipeline

In [4]:
### 1. Load the processed Maps and the data_cube_32 ###

path_to_websky = '/Users/joanribot/HEAVY_STUFF/TFM_data/Processed_Maps/'

print("Loading the y_map, cmb_map and data_cube (4 layers: CMB + tSZ + noise + mu-distorted tSZ) from file...")

# A. Load the Data Cube (The 'Observed' (simulated) data)
data_cube = hp.read_map(path_to_websky + "data_cube_32_9freq.fits", field=None).astype(np.float32)

# B. Load the Truth Maps (For validation and residuals)
y_map = hp.read_map(path_to_websky + "y_map.fits").astype(np.float32)
cmb_map = hp.read_map(path_to_websky + "cmb_map.fits").astype(np.float32)

print(f"Data Cube loaded. Shape: {data_cube.shape}")
print(f"{data_cube.shape[0]} frequencies: {frequencies} with 4 layer maps of {data_cube.shape[1]} pixels as Nside = {nside_work}.")
# Should be (9, 50331648) for Nside 2048

Loading the y_map, cmb_map and data_cube (4 layers: CMB + tSZ + noise + mu-distorted tSZ) from file...
Data Cube loaded. Shape: (9, 50331648)
9 frequencies: [30, 44, 70, 100, 143, 217, 353, 545, 857] with 4 layer maps of 50331648 pixels as Nside = 2048.


In [ ]:
### 1.5 Load the processed Maps and the data_cube_64 ###

path_to_websky = '/Users/joanribot/HEAVY_STUFF/TFM_data/Processed_Maps/'

print("Loading the y_map, cmb_map and data_cube (4 layers: CMB + tSZ + noise + mu-distorted tSZ) from file...")

# A. Load the Data Cube (The 'Observed' (simulated) data)
data_cube = hp.read_map(path_to_websky + "data_cube_64_9freq.fits", field=None).astype(np.float64)

# B. Load the Truth Maps (For validation and residuals)
y_map = hp.read_map(path_to_websky + "y_map.fits").astype(np.float64)
cmb_map = hp.read_map(path_to_websky + "cmb_map.fits").astype(np.float64)

print(f"Data Cube loaded. Shape: {data_cube.shape}")
print(f"{data_cube.shape[0]} frequencies: {frequencies} with 4 layer maps of {data_cube.shape[1]} pixels as Nside = {nside_work}.")
# Should be (9, 50331648) for Nside 2048

In [5]:
### 2. Define the tSZ frequency scaling function g(nu) and Taylor expansion h(nu) ###

def get_physics_constants(nu):
    """Calculate the physics constants needed for tSZ scaling."""
    T_cmb = 2.7255     # CMB temperature in K
    k_B = 1.380649e-23 # Boltzmann constant in J/K
    h = 6.62607015e-34 # Planck constant in J*s
    x = (h * nu * 1e9) / (k_B * T_cmb) # Dimensionless frequency
    return x

def get_tsz_g(nu):
    """Calculate the tSZ frequency scaling g(nu) in dimensionless units."""
    x = get_physics_constants(nu)
    g_nu = x * (np.exp(x) + 1) / (np.exp(x) - 1) - 4 # or alternatively: g_nu = x * (1 / np.tanh(x / 2)) - 4
    return g_nu

def get_tsz_h(nu):
    """Calculate the tSZ Taylor expansion frequency scaling h(nu) in dimensionless units."""
    x = get_physics_constants(nu)
    h_nu = (-x/2) * ( 1 / ( (np.exp(x) - np.exp(-x)) / 2 )**2 ) # or alternatively: h_nu = (-x/2) * (1 / np.sinh(x/2)**2)
    #h_nu = (-x/2) * (1 / np.sinh(x/2)**2)
    return h_nu

## CILC

### Outline 

1. CILC method definition: Pixel CILC (P-CILC) & Harmonic CILC (H-CILC).

2. Processing for each signal:

| **Subsection** | __Signal__ | **SED** | **Extracted map** |
| :---: | :---: | :---: | :---: |
| 2.1. | tSZ | $g(\nu) = x \coth \left( \frac{x}{2} \right) - 4 $ | $y$-map |
| 2.2. | CMB | $a=[1,...,1]$ | $CMB$-map |
| 2.3. | $\mu$-distorted tSZ | $h(\nu) = - \frac{x}{2} \frac{1}{\sinh^2\left( \frac{x}{2} \right)}$ | $\mu y$-map |

3. Plotting two views for every map:
- Zoomed view.
- Mollweide view.

4. Extra plots.

5. TT map: $\mu y$ Vs. $y$ for $\mu$ extraction.

## 1. CILC method definition

### Old ILC definitions

In [ ]:
def run_pixel_ilc(data_cube, vector_muK): ### UNITLESS MAP OUTPUT + WEIGHTS ###

    # C definition
    n_freq, n_pix = data_cube.shape
    C = (1.0 / n_pix) * (data_cube @ data_cube.T) # IS IT THAT d_i (p') * d_j (p') = data_cube @ data_cube.T
    C_inv = np.linalg.pinv(C)

    # Calculate the weights
    weights = (C_inv @ vector_muK) / (vector_muK.T @ C_inv @ vector_muK)

    return weights @ data_cube, weights

# weights @ data_cube IS UNITLESS AS [weights] = 1/g^t * data_cube = 1/T_cmb * T_cmb
# [g] = T_cmb_muK
# [data_cube] = T_cmb_muK

In [ ]:
def run_harmonic_ilc_with_weights(data_cube, vector_muK, l_bins): ### UNITLESS MAP OUTPUT + WEIGHTS PER BIN  ###
    """Advanced Harmonic-Space ILC (Multi-scale)"""
    n_freq, n_pix = data_cube.shape
    nside = hp.npix2nside(n_pix)
    lmax = 3*nside-1
    alms_list = [hp.map2alm(m, lmax=lmax) for m in data_cube]
    final_alms = np.zeros(hp.Alm.getsize(lmax), dtype=np.complex128)
    ell, _ = hp.Alm.getlm(lmax)
    
    # NEW: Matrix to store weights for each bin
    weights_per_bin = []

    for i in range(len(l_bins)-1):
        mask = (ell >= l_bins[i]) & (ell < l_bins[i+1])
        bin_alms = np.array([alms[mask] for alms in alms_list])
        cov_bin = np.real(np.conj(bin_alms) @ bin_alms.T) / np.sum(mask)
        inv_cov = np.linalg.pinv(cov_bin)
        
        # Calculate weights for THIS specific scale
        w_bin = (inv_cov @ vector_muK) / (vector_muK.T @ inv_cov @ vector_muK)
        weights_per_bin.append(w_bin) # Store them
        
        final_alms[mask] = w_bin @ bin_alms
        
    return hp.alm2map(final_alms, nside=nside), np.array(weights_per_bin)

### CILC definitions (after on-line meeting)

In [ ]:
def run_pixel_cilc(data_cube, a, b): ### UNITLESS MAP OUTPUT + WEIGHTS ### --- ### it doesn't work as the () @ () product is not working... ###

    # R definition
    n_freq, n_pix = data_cube.shape
    R = (1.0 / n_pix) * (data_cube @ data_cube.T) # IS IT THAT d_i (p') * d_j (p') = data_cube @ data_cube.T
    R_inv = np.linalg.pinv(R)

    # Calculate the weights
    #w_pixel = (R_inv @ a) / (a.T @ R_inv @ a) - (R_inv @ b) / (b.T @ R_inv @ b) # se la fuma fortet
    w_pixel = ( (b.T @ R_inv @ b) @ a.T @ R_inv - (a.T @ R_inv @ b) @ b.T @ R_inv ) / ( (a.T @ R_inv @ a) @ (b.T @ R_inv @ b) - (a.T @ R_inv @ b)**2 )

    return w_pixel @ data_cube, w_pixel

# weights @ data_cube IS UNITLESS AS [weights] = 1/g^t * data_cube = 1/T_cmb * T_cmb
# [a],[b] = T_cmb_muK
# [data_cube] = T_cmb_muK

### Revisar definició de harmonic... crec que calcula es weights com li passa pes forro

## 2. CILC implementation

In [6]:
### Define the mixing vectors (SEDs) in muK: g_vector_muK, h_vector_muK, a_cmb_muK ###

# Define g_vector and h_vector in muK units for the frequencies in the data cube
# Mixing vectors (SEDs) in muK
g_vector_muK = np.array([get_tsz_g(nu) for nu in frequencies]) * T_cmb_muK # IT HAS TO BE MULTIPLIED BY T_CMB * 10^6 or T_CMB_MU_K
h_vector_muK = np.array([get_tsz_h(nu) for nu in frequencies]) * T_cmb_muK

# CMB mixing vector in muK units (constant across frequencies) --- NOT NEEDED SO FAR...
#a_cmb_muK = np.ones(len(frequencies)) * T_cmb_muK

In [7]:
### Check lengths of the vectors and frequencies ###
print("Length of g_vector_muK: ", len(g_vector_muK)) # Should be 9, one for each frequency
print("Length of h_vector_muK: ", len(h_vector_muK)) # Should be 9, one for each frequency
#print("Length of a_cmb_muK: ", len(a_cmb_muK)) # Should be 9, one for each frequency
print("Length of frequencies: ", len(frequencies)) # Should be 9, one for each frequency

Length of g_vector_muK:  9
Length of h_vector_muK:  9
Length of frequencies:  9


### New P-CILC definition...

In [9]:
def run_pixel_cilc_2(data_cube, a, b):
    # R definition
    n_freq, n_pix = data_cube.shape
    # Ensure double precision for tiny signal extraction
    R = (1.0 / n_pix) * (data_cube.astype(np.float64) @ data_cube.T.astype(np.float64))
    R_inv = np.linalg.pinv(R)

    # Scalar terms for the denominator and numerator
    # We use .item() or np.dot to ensure these are scalars, not 1x1 arrays
    iRa = R_inv @ a
    iRb = R_inv @ b
    
    aRa = a.T @ iRa
    bRb = b.T @ iRb
    aRb = a.T @ iRb

    # Corrected formula with explicit multiplication (*)
    denominator = (aRa * bRb) - (aRb**2)
    
    # Numerator calculation
    numerator = (bRb * iRa.T) - (aRb * iRb.T)
    
    w_pixel = numerator / denominator

    # Apply weights: w_pixel is a row vector, data_cube is (n_freq, n_pix)
    return w_pixel @ data_cube, w_pixel

In [11]:
### CALL THE P-CILC METHOD FOR mu-distorted tSZ --- run_pixel_cilc_2 ###

# True mu-distorted tSZ signal in dimensionless units
muy_map = mu * y_map

# Pixel CILC
muy_hat_pixel, w_pixel = run_pixel_cilc_2(data_cube, h_vector_muK, g_vector_muK) # Want to preserve h and null g

# Calculate the residual in dimensionless mu*y units (Truth - Reconstruction)
muy_residual_pixel = muy_map - muy_hat_pixel

# Validation: Both should be order 10^-13 (since mu is 10^-8 and y is 10^-5)
print(f"Truth max: {np.max(muy_map):.2e}")
print(f"Reco max: {np.max(muy_hat_pixel):.2e}")
print(f"Residual max: {np.max(muy_residual_pixel):.2e}")
print("WATCH!!! THERE IS A PROBLEM WITH RECO...")

Truth max: 3.99e-07
Reco max: 6.66e-05
Residual max: 6.41e-05
WATCH!!! THERE IS A PROBLEM WITH RECO...


In [ ]:
### H-ILC (with weights) ###
l_bins = [2, 25, 50, 75, 100, 150, 200, 300, 500, 1000]

# Mathieu's paper uses these l_bins: Delta_l = 50 for l < 2075 and Delta_l = 20 for l > 2075...
l_bins_Mathieu = [2, 50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 850, 900, 950, 1000,
               1050, 1100, 1150, 1200, 1250, 1300, 1350, 1400, 1450, 1500, 
               1550, 1600, 1650, 1700, 1750, 1800, 1850, 1900, 
               1950, 2000] + list(range(2050,2075+20,20)) + list(range(2100,3000+20,20))
"""
Si emplees aquest s'ordinador se mor, això seria per fer servir es CLUSTER
"""

# Calculate the harmonic ILC reconstruction and get the weights for each bin
y_hat_harmonic, weights_harmonic_bins = run_harmonic_cilc_with_weights(data_cube, h_vector_muK, g_vector_muK, l_bins) # Want to preserve h and null g

# Residual map in muK units for harmonic ILC
y_residual_harmonic = y_map - y_hat_harmonic

# Validation: Both should be order 10^-5
print(f"Truth max: {np.max(y_map):.2e}")
print(f"Reco max: {np.max(y_hat_harmonic):.2e}")
print(f"Residual max: {np.max(y_residual_harmonic):.2e}")

### Have you seen the new CILC definition yet?

In [9]:
def run_pixel_cilc_3(data_cube, a_vec, b_vec):
    n_freq, n_pix = data_cube.shape
    # IMPORTANT: R must be calculated in float64 for the tiny mu signal
    R = (1.0 / n_pix) * (data_cube.astype(np.float64) @ data_cube.T.astype(np.float64))
    R_inv = np.linalg.pinv(R)

    # Constraint Matrix A = [a, b]
    A = np.column_stack((a_vec, b_vec))
    # Target vector e = [1, 0] -> Preserve 'a', Null 'b'
    e = np.array([1.0, 0.0])

    # Weight formula: w = R_inv @ A @ (A.T @ R_inv @ A)^-1 @ e
    # This is the matrix version
    weights = R_inv @ A @ np.linalg.inv(A.T @ R_inv @ A) @ e
    
    return weights @ data_cube, weights

In [10]:
### CALL THE P-CILC METHOD FOR mu-distorted tSZ --- run_pixel_cilc_3 ###

# True mu-distorted tSZ signal in dimensionless units
muy_map = mu * y_map

# Pixel CILC
muy_hat_pixel, w_pixel = run_pixel_cilc_3(data_cube, h_vector_muK, g_vector_muK) # Want to preserve h and null g

# Calculate the residual in dimensionless mu*y units (Truth - Reconstruction)
muy_residual_pixel = muy_map - muy_hat_pixel

# Validation: Both should be order 10^-13 (since mu is 10^-8 and y is 10^-5)
print(f"Truth max: {np.max(muy_map):.2e}")
print(f"Reco max: {np.max(muy_hat_pixel):.2e}")
print(f"Residual max: {np.max(muy_residual_pixel):.2e}")
print("WATCH!!! THERE IS A PROBLEM WITH RECO...")

Truth max: 3.99e-07
Reco max: 6.66e-05
Residual max: 6.41e-05
WATCH!!! THERE IS A PROBLEM WITH RECO...


In [11]:
### Check the constraints with the weights obtained from the final pixel CILC

print(f"Preservation Check (H): {np.dot(w_pixel, h_vector_muK):.4f}") # MUST BE 1.000
print(f"Nulling Check (G):       {np.dot(w_pixel, g_vector_muK):.4e}") # MUST BE < 1e-12

Preservation Check (H): 1.0000
Nulling Check (G):       -2.2204e-16


## 2.1. tSZ signal: $g(\nu) \rightarrow$ ($y$-map)

Execute both P-CILC & H-CILC methods separetly for the seek of time. Harmonic-space is more time consuming as more bins are considered.

In [ ]:
### CALL THE P-CILC METHOD FOR THE tSZ --- run_pixel_cilc_2 ###

y_hat_pixel, w_pixel = run_pixel_cilc_2(data_cube, g_vector_muK, h_vector_muK) # Want to preserve g and null h

y_residual_pixel = y_map - y_hat_pixel

print(f"Truth max: {np.max(y_map):.2e}")
print(f"Reco max: {np.max(y_hat_pixel):.2e}")
print(f"Residual max: {np.max(y_residual_pixel):.2e}")

In [ ]:
### ANALYSIS WEIGHTS FOR PIXEL CILC ###

df_weights_pixel_cilc = pd.DataFrame({
    'Freq (GHz)': frequencies,
    'Weight': w_pixel,
    'g(nu) * T_cmb_muK': g_vector_muK,
    'h(nu) * T_cmb_muK': h_vector_muK
})
# Transpose it for horizontal display
df_horizontal_pixel_cilc = df_weights_pixel_cilc.set_index('Freq (GHz)').T
pd.set_option('display.expand_frame_repr', False) 
pd.set_option('display.max_columns', None)
print("--- CILC Analysis Weights ---")
print(df_horizontal_pixel_cilc)
print()

# Verify the constraint is satisfied
constraint_check_pixel_cilc_g = np.sum(w_pixel * g_vector_muK)
constraint_check_pixel_cilc_h = np.sum(w_pixel * h_vector_muK)
w_pixel_sum = np.sum(w_pixel)
print(f"Constraint Check (g): {constraint_check_pixel_cilc_g:.4f}")
print(f"Constraint Check (h): {constraint_check_pixel_cilc_h:.4f}")
print(f"Sum of Weights: {w_pixel_sum:.4f}")
#print("Great! Since the CMB has the same value in every channel, the ILC 'subtracts' the low-frequency channels from the high-frequency ones to make the CMB disappear.")

In [ ]:
### CHECK THE CONSTRAINTS FOR PIXEL CILC ###

print(f"Check Mu-Preservation (should be 1): {np.dot(w_pixel, h_vector_muK):.4f}")
print(f"Check tSZ-Nulling (should be 0):     {np.dot(w_pixel, g_vector_muK):.4e}")

In [ ]:
### DOUBLE CHECK THE CONSTRAINTS ###

print(f"Constraint a (should be 1): {w_pixel @ h_vector_muK:.4f}")
print(f"Constraint b (should be 0): {w_pixel @ g_vector_muK:.4f}")

## 3.1. P-CILC & H-CILC: $y$-maps

In [ ]:
### A.1. PIXEL CILC ZOOMED MAPS ###

params_zoom = {'rot': [0, 0], 'reso': 1.5, 'xsize': 500, 'hold': True, 'cmap': 'jet', 'notext': False, 'min': -5*10**-6, 'max': 5*10**-6} # Added 'coord': 'G' to specify Galactic coordinates
#params_zoom = {'rot': [0, 0], 'reso': 1.5, 'xsize': 500, 'hold': True, 'cmap': 'jet', 'notext': True}

# --- Plotting ---
fig_harm = plt.figure(figsize=(15, 5))

ax1 = plt.subplot(1, 3, 1)
hp.gnomview(y_map, title=rf"Input $y$", **params_zoom)
hp.graticule(dpar=0.5, dmer=0.5, color='white', alpha=0.3)
plt.plot(0, 0, marker='+', color='red', markersize=10, label='Center')

ax2 = plt.subplot(1, 3, 2)
hp.gnomview(y_hat_pixel, title="P-CILC", **params_zoom)
hp.graticule(dpar=0.5, dmer=0.5, color='white', alpha=0.3)
plt.plot(0, 0, marker='+', color='red', markersize=10, label='Center')

ax3 = plt.subplot(1, 3, 3)
hp.gnomview(y_residual_pixel, title="Residual (P-CILC - Input)", **params_zoom)
hp.graticule(dpar=0.5, dmer=0.5, color='white', alpha=0.3)
plt.plot(0, 0, marker='+', color='red', markersize=10, label='Center')

plt.suptitle("Zoomed maps (unitless)")
plt.savefig("Figures/CILC/y_maps/P-CILC_zoomed_maps.pdf", dpi=300, bbox_inches='tight')
plt.show()

## 2.2. $\mu$-distorted tSZ: $f(\nu) \rightarrow$ ($\mu y$-map)

In [ ]:
### CALL THE P-CILC METHOD FOR mu-distorted tSZ --- run_pixel_cilc_2 ###

# True mu-distorted tSZ signal in dimensionless units
muy_map = mu * y_map

# Pixel CILC
#muy_hat_pixel, w_pixel = run_pixel_cilc(data_cube, seds, targets)
muy_hat_pixel, w_pixel = run_pixel_cilc_2(data_cube, h_vector_muK, g_vector_muK) # Want to preserve h and null g

# Calculate the residual in dimensionless mu*y units (Truth - Reconstruction)
muy_residual_pixel = muy_map - muy_hat_pixel

"""
# As y_map is also unitless, I will leave both unitless. However, I could also convert both to muK units and have a 10^5 scale factor.
muy_map_muK = muy_map * T_cmb_muK
residual_pixel_muK = muy_map_muK - muy_hat_pixel * T_cmb_muK
"""

# Validation: Both should be order 10^-13 (since mu is 10^-8 and y is 10^-5)
print(f"Truth max: {np.max(muy_map):.2e}")
print(f"Reco max: {np.max(muy_hat_pixel):.2e}")
print(f"Residual max: {np.max(muy_residual_pixel):.2e}")
print("WATCH!!! THERE IS A PROBLEM WITH RECO...")

In [ ]:
### ANALYSIS WEIGHTS FOR PIXEL CILC ###

df_weights_pixel_cilc = pd.DataFrame({
    'Freq (GHz)': frequencies,
    'Weight': w_pixel,
    'g(nu) * T_cmb_muK': g_vector_muK,
    'h(nu) * T_cmb_muK': h_vector_muK
})
# Transpose it for horizontal display
df_horizontal_pixel_cilc = df_weights_pixel_cilc.set_index('Freq (GHz)').T
pd.set_option('display.expand_frame_repr', False) 
pd.set_option('display.max_columns', None)
print("--- CILC Analysis Weights ---")
print(df_horizontal_pixel_cilc)
print()

# Verify the constraint is satisfied
constraint_check_pixel_cilc_g = np.sum(w_pixel * g_vector_muK)
constraint_check_pixel_cilc_h = np.sum(w_pixel * h_vector_muK)
w_pixel_sum = np.sum(w_pixel)
print(f"Constraint Check (g): {constraint_check_pixel_cilc_g:.4f}")
print(f"Constraint Check (h): {constraint_check_pixel_cilc_h:.4f}")
print(f"Sum of Weights: {w_pixel_sum:.4f}")
#print("Great! Since the CMB has the same value in every channel, the ILC 'subtracts' the low-frequency channels from the high-frequency ones to make the CMB disappear.")

In [12]:
### CHECK THE CONSTRAINTS FOR PIXEL CILC ###

print(f"Check Mu-Preservation (should be 1): {np.dot(w_pixel, h_vector_muK):.4f}")
print(f"Check tSZ-Nulling (should be 0):     {np.dot(w_pixel, g_vector_muK):.4e}")

Check Mu-Preservation (should be 1): 1.0000
Check tSZ-Nulling (should be 0):     -2.2204e-16


In [13]:
### DOUBLE CHECK THE CONSTRAINTS ###

print(f"Constraint a (should be 1): {w_pixel @ h_vector_muK:.4f}")
print(f"Constraint b (should be 0): {w_pixel @ g_vector_muK:.4f}")

Constraint a (should be 1): 1.0000
Constraint b (should be 0): -0.0000


## Calibrated g vector

In [21]:
### CHECK THE CALIBRATION OF THE g_vector BASED ON THE MAX VALUES IN THE DATA CUBE ###

g_vector_calibrated = np.array([np.max(data_cube[i]) / np.max(y_map) for i in range(len(frequencies))])

print("Calibrated g_vector based on max values:")
print(g_vector_calibrated)
print("Maximum value of calibrated g_vector:", np.max(g_vector_calibrated))
print()
print("Original g_vector:")
print(g_vector_muK)
print("Maximum value of original g_vector:", np.max(g_vector_muK))

Calibrated g_vector based on max values:
[2.9572425e+06 2.9809618e+06 3.0572895e+06 3.1222792e+06 3.0994162e+06
 3.2094210e+06 6.4129475e+06 1.4870955e+07 2.9706370e+07]
Maximum value of calibrated g_vector: 2.970637e+07

Original g_vector:
[-5.32482325e+06 -5.18100961e+06 -4.77771592e+06 -4.11032770e+06
 -2.83551138e+06 -2.11910750e+04  6.10714400e+06  1.52574296e+07
  3.02275361e+07]
Maximum value of original g_vector: 30227536.121719938


In [ ]:
### RUN THE P-CILC METHOD FOR mu-distorted tSZ WITH THE CALIBRATED g_vector --- run_pixel_cilc_3 ###

muy_hat_pixel, weights = run_pixel_cilc_3(data_cube, h_vector_muK, g_vector_calibrated) # Want to preserve h and null the calibrated g

muy_residual_pixel = muy_map - muy_hat_pixel

# Validation: Both should be order 10^-13 (since mu is 10^-8 and y is 10^-5)
print(f"Truth max: {np.max(muy_map):.2e}")
print(f"Reco max: {np.max(muy_hat_pixel):.2e}")
print(f"Residual max: {np.max(muy_residual_pixel):.2e}")
print("WATCH!!! THERE IS A PROBLEM WITH RECO...")

In [19]:
# Validation: Both should be order 10^-13 (since mu is 10^-8 and y is 10^-5)
print(f"Truth max: {np.max(muy_map):.2e}")
print(f"Reco max: {np.max(muy_hat_pixel):.2e}")
print(f"Residual max: {np.max(muy_residual_pixel):.2e}")
print("WATCH!!! THERE IS A PROBLEM WITH RECO...")

Truth max: 3.99e-07
Reco max: 2.85e-04
Residual max: 6.41e-05
WATCH!!! THERE IS A PROBLEM WITH RECO...


In [18]:
### ANALYSIS WEIGHTS FOR PIXEL CILC WITH CALIBRATED g_vector ###

df_weights_pixel_cilc_calibrated = pd.DataFrame({
    'Freq (GHz)': frequencies,
    'Weight': weights,
    'Calibrated g(nu)': g_vector_calibrated,
    'h(nu) * T_cmb_muK': h_vector_muK
})
# Transpose it for horizontal display
df_horizontal_pixel_cilc_calibrated = df_weights_pixel_cilc_calibrated.set_index('Freq (GHz)').T
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)
print("--- CILC Analysis Weights with Calibrated g_vector ---")
print(df_horizontal_pixel_cilc_calibrated)
print()

--- CILC Analysis Weights with Calibrated g_vector ---
Freq (GHz)                  30            44            70            100           143           217           353           545           857
Weight            -4.331204e-07 -6.567911e-08  9.496472e-08  1.247808e-07  1.172619e-07  9.221829e-08  5.699941e-08  2.164030e-08 -1.851667e-08
Calibrated g(nu)   2.957242e+06  2.980962e+06  3.057290e+06  3.122279e+06  3.099416e+06  3.209421e+06  6.412948e+06  1.487096e+07  2.970637e+07
h(nu) * T_cmb_muK -2.352555e+06 -1.445480e+06 -6.820575e+05 -3.011515e+05 -9.037758e+04 -1.000300e+04 -1.351926e+02 -2.415403e-01 -6.421383e-06



In [17]:
### CHECK THE CONSTRAINTS FOR PIXEL CILC WITH CALIBRATED VECTOR ###

print(f"Check Mu-Preservation (should be 1): {np.dot(weights, h_vector_muK):.4f}")
print(f"Check tSZ-Nulling (should be 0):     {np.dot(weights, g_vector_calibrated):.4e}")

Check Mu-Preservation (should be 1): 1.0000
Check tSZ-Nulling (should be 0):     3.3307e-16


## 3.2. P-ILC & H-ILC: $\mu y$-maps

In [ ]:
### A.1. PIXEL CILC ZOOMED MAPS ###

params_zoom = {'rot': [0, 0], 'reso': 1.5, 'xsize': 500, 'hold': True, 'cmap': 'jet', 'notext': False, 'min': -5*10**-13, 'max': 5*10**-13} # Added 'coord': 'G' to specify Galactic coordinates
#params_zoom = {'rot': [0, 0], 'reso': 1.5, 'xsize': 500, 'hold': True, 'cmap': 'jet', 'notext': True}

# --- Plotting ---
fig_harm = plt.figure(figsize=(15, 5))

ax1 = plt.subplot(1, 3, 1)
hp.gnomview(muy_map, title=rf"Input $\mu$-distorted $y$", **params_zoom)
hp.graticule(dpar=0.5, dmer=0.5, color='white', alpha=0.3)
plt.plot(0, 0, marker='+', color='red', markersize=10, label='Center')

ax2 = plt.subplot(1, 3, 2)
hp.gnomview(muy_hat_pixel, title="P-CILC", **params_zoom)
hp.graticule(dpar=0.5, dmer=0.5, color='white', alpha=0.3)
plt.plot(0, 0, marker='+', color='red', markersize=10, label='Center')

ax3 = plt.subplot(1, 3, 3)
hp.gnomview(muy_residual_pixel, title="Residual (P-CILC - Input)", **params_zoom)
hp.graticule(dpar=0.5, dmer=0.5, color='white', alpha=0.3)
plt.plot(0, 0, marker='+', color='red', markersize=10, label='Center')

plt.suptitle("Zoomed maps (unitless)")
plt.savefig("Figures/CILC/mu_y_maps/P-CILC_zoomed_maps.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
### B.1. HARMONIC ILC ZOOMED MAPS ###

params_zoom = {'rot': [0, 0], 'reso': 1.5, 'xsize': 500, 'hold': True, 'cmap': 'jet', 'notext': True, 'min': -5*10**-13, 'max': 5*10**-13}
#params_zoom = {'rot': [0, 0], 'reso': 1.5, 'xsize': 600, 'hold': True, 'cmap': 'jet', 'notext': True}

# --- Plotting ---
fig_harm = plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
#hp.gnomview(y_map_muK, title="Harmonic ILC: Full Sky (y)", **params_zoom)
hp.gnomview(muy_map, title="Input $y$", **params_zoom)
hp.graticule(dpar=0.5, dmer=0.5, color='white', alpha=0.3)
plt.plot(0, 0, marker='+', color='red', markersize=10, label='Center')

plt.subplot(1, 3, 2)
#hp.gnomview(y_hat_harmonic_muK, title="H-ILC", **params_zoom) ### NEXT TIME U COMPILE REMEMBER TO SWAP, I JUST SWAPED NAMES (_muK)
hp.gnomview(muy_hat_harmonic, title="H-ILC", **params_zoom)
hp.graticule(dpar=0.5, dmer=0.5, color='white', alpha=0.3)
plt.plot(0, 0, marker='+', color='red', markersize=10, label='Center')

plt.subplot(1, 3, 3)
#hp.gnomview(y_residual_harmonic_muK, title="Harmonic ILC: Residual (Truth - Reco)", **params_zoom)
hp.gnomview(muy_residual_harmonic, title="Residual (H-ILC - Input)", **params_zoom)
hp.graticule(dpar=0.5, dmer=0.5, color='white', alpha=0.3)
plt.plot(0, 0, marker='+', color='red', markersize=10, label='Center')

plt.suptitle("Zoomed maps (unitless)")
plt.savefig("Figures/CILC/mu_y_maps/H-CILC_zoomed_maps.pdf", dpi=300, bbox_inches='tight')
plt.show()

## 5. T-T plot: $\mu y$ Vs. $y$ for $\mu$ extraction

### 5.1. Calculating cluster galactic coordinates: (l,b)

In [ ]:
def get_brightest_cluster_coords(y_map, n_clusters=500, min_dist_deg=1.0):
    """Finds the coordinates of the n brightest clusters."""
    # Create a copy to avoid modifying original
    temp_map = np.copy(y_map)
    nside = hp.npix2nside(len(temp_map))
    coords = []
    
    for i in range(n_clusters):
        # Find pixel with maximum value
        pix_max = np.argmax(temp_map)
        
        # Convert to Galactic coordinates
        theta, phi = hp.pix2ang(nside, pix_max)
        lon = np.degrees(phi)
        lat = 90 - np.degrees(theta)
        coords.append((lon, lat))
        
        # Zero out the surrounding area so we don't pick the same cluster
        # A 1-degree radius is usually enough for WebSky clusters
        vec = hp.pix2vec(nside, pix_max)
        disc_pix = hp.query_disc(nside, vec, radius=np.radians(min_dist_deg))
        temp_map[disc_pix] = 0
        
    return coords

In [ ]:
### EXTRACTING THE COORDINATES OF THE BRIGHTEST CLUSTERS ###
cluster_coords = get_brightest_cluster_coords(y_map)
print("Coordinates of the 500 brightest clusters.")
print("Only the first 10 are shown for brevity (lon, lat) in degrees):")
for i, (lon, lat) in enumerate(cluster_coords[:10]):  # Print only the first 10 for brevity
    print(f"{i+1}: (lon: {lon:.2f}, lat: {lat:.2f})")

### 5.2. Stacking clusters from $y$ and $\mu y$ maps

In [ ]:
def stack_clusters(hp_map, coords, reso_arcmin=1.5, xsize=100):
    """Stacks small patches around specified coordinates."""
    stacked_patch = np.zeros((xsize, xsize))
    
    for lon, lat in coords:
        # Project a small area into a flat 2D patch
        patch = hp.gnomview(hp_map, rot=[lon, lat], reso=reso_arcmin, 
                            xsize=xsize, return_projected_map=True, no_plot=True)
        stacked_patch += patch
        
    return stacked_patch / len(coords)

In [ ]:
### STACKING THE CILC SIGNALS AROUND THE BRIGHTEST CLUSTERS ###

# Extract stacked signals from your CILC reconstructions
# (or use the truth maps first to verify the method)
stack_y = stack_clusters(y_hat_pixel, cluster_coords)
stack_muy = stack_clusters(muy_hat_pixel, cluster_coords)

In [ ]:
### STACKING THE REAL SIGNALS AROUND THE BRIGHTEST CLUSTERS ###

y_map = y_map # The true tSZ signal in dimensionless units
muy_map = mu * y_map # The true mu-distorted tSZ signal in dimensionless units

# Extract stacked signals from your ILC reconstructions
# (or use the truth maps first to verify the method)
stack_y_real = stack_clusters(y_map, cluster_coords)
stack_muy_real = stack_clusters(muy_map, cluster_coords)

### 5.3. Temperature-Temperature plot: $\mu y$ Vs. $y$ and linear regression - the slope represents $\mu$

In [ ]:
def plot_tt_correlation(stack_y, stack_muy):
    # Flatten the 2D patches into 1D arrays for regression
    x_data = stack_y.flatten()
    y_data = stack_muy.flatten()
    
    # Linear Regression
    slope, intercept, r_value, p_value, std_err = stats.linregress(x_data, y_data)
    
    # Plotting
    plt.figure(figsize=(8, 6))
    plt.scatter(x_data, y_data, alpha=0.5, s=5, label='Cluster Pixels')
    
    # Plot the regression line
    line = slope * x_data + intercept
    #plt.plot(x_data, line, color='red', label=f'Fit: slope={slope:.2e}')
    plt.plot(x_data, line, color='red', label=f'y = {slope:.2e}x + {intercept:.2e}')

    # Limit the axes to focus on the cluster signal range
    #plt.xlim(0, 1e-5)
    #plt.ylim(0, 1e-5)

    plt.xlabel(rf'Stacked $y$ Signal')
    plt.ylabel(rf'Stacked $\mu y$ Signal')
    plt.title(rf'T-T Plot: $\mu y$ Vs. $y$ Correlation')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("Figures/ILC/TT_plot.pdf", dpi=300, bbox_inches='tight')
    plt.show()
    
    return slope, intercept, r_value, p_value, std_err

In [ ]:
### PLOTTING THE ILC T-T CORRELATION AND EXTRACTING THE MEASURED MU VALUE ###

measured_mu, intercept, r_value, p_value, std_err = plot_tt_correlation(stack_y, stack_muy)
print(f"--- Statistics ---")
print(rf"y = {measured_mu:.2e} * x + {intercept:.2e}")
print(f"R-squared: {r_value**2:.4f}, p-value: {p_value:.2e}, std_err: {std_err:.2e}")
print("----------------------")
print(f"Measured mu = {measured_mu:.2e}")
print(f"Input    mu = 2.00e-08")

In [ ]:
### PLOTTING THE REAL T-T CORRELATION AND EXTRACTING THE MEASURED MU VALUE ###

measured_mu, intercept, r_value, p_value, std_err = plot_tt_correlation(stack_y_real, stack_muy_real)
print(f"--- Statistics ---")
print(rf"y = {measured_mu:.2e} * x + {intercept:.2e}")
print(f"R-squared: {r_value**2:.4f}, p-value: {p_value:.2e}, std_err: {std_err:.2e}")
print("----------------------")
print(f"Measured mu = {measured_mu:.2e}")
print(f"Input    mu = 2.00e-08")